# 04 · One-stage anchor-free detector: YOLOv8

YOLOv8 (Ultralytics) predicts, at every location of the stride-8/16/32 grids, the distances from that location to the four box edges (as discrete distributions, DFL) plus class scores; there are no anchor boxes. Positives are chosen by the **Task-Aligned Assigner** (TOOD): among grid points whose centre lies inside a GT box, the top-k by `score^α · IoU^β` become positives.

Protocol: same frames, `imgsz`, epochs and evaluator as notebooks 02–03. Augmentation is reduced to horizontal flip (`ultra.PROTOCOL_AUG`) so that the comparison is about the model, not the augmentation.

Experiments: **Y0** yolov8s → **Y1** + P2 (stride-4) head → **Y2** input-resolution search {1024, 1280, 1536} on the better architecture → **Y3** label assignment (TAL top-k 10 → 13).

## 1. Setup, protocol and dataset export

In [ ]:
import os, sys, json, shutil
from pathlib import Path

REPO = Path("/kaggle/working/repo") if Path("/kaggle").exists() else Path("..").resolve()
if Path("/kaggle").exists() and not REPO.exists():
    !git clone https://github.com/<user>/<repo>.git {REPO}
    !pip install -q faster-coco-eval ultralytics==8.3.*
sys.path.insert(0, str(REPO / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches, cv2
import vdet, parallel, ultra
from vdet import CLASSES

vdet.seed_everything()
ROOT = vdet.find_data_root()
OUT = Path("/kaggle/working") if Path("/kaggle").exists() else REPO / "outputs"
RUNS = OUT / "runs"; RUNS.mkdir(parents=True, exist_ok=True)
RESULT_DIRS = [RUNS, REPO / "results"]   # own runs + runs committed by the teammate
P = vdet.PROTOCOL

# restore finished runs from earlier versions of this notebook (attached as Kaggle input)
for src in (Path("/kaggle/input").glob("*/runs/*") if Path("/kaggle/input").exists() else []):
    if (src / "config.json").exists() and not (RUNS / src.name).exists():
        shutil.copytree(src, RUNS / src.name); print("restored", src.name)

In [ ]:
imgs, anns = vdet.load_all(ROOT)
folds = pd.read_csv(OUT / "folds.csv") if (OUT / "folds.csv").exists() else vdet.make_folds(imgs, P["n_folds"])
folds.to_csv(OUT / "folds.csv", index=False)   # the parallel runner reads the split from this file
train_frames, val_frames = vdet.protocol_frames(imgs, folds, P)
train_gt = anns[anns.key.isin(train_frames.key) & (anns.iscrowd == 0)]
print(json.dumps(P))
print(f"train: {len(train_frames)} frames | val: {len(val_frames)} frames / {val_frames.clip_uid.nunique()} clips")

# Ultralytics needs files on disk: symlinks + masked copies of frames with ignored regions
data_yaml = vdet.build_yolo_dataset(imgs, anns, folds, val_fold=P["val_fold"], out_dir=OUT / "yolo_fold0",
                                    step=P["step"], val_step=P["val_step"], mask_ignore=True)

## 2. How locations are assigned without anchors — and what it means for tiny objects

A box can only get positives from grid points whose centre falls inside it. The table counts how many such points a training box has on each stride (boxes rescaled to `imgsz`).

In [ ]:
sc = (P["imgsz"] / np.maximum(train_gt.width, train_gt.height)).values
w, h = train_gt.w.values * sc, train_gt.h.values * sc
rows = []
for s in [4, 8, 16, 32]:
    pts = np.floor(w / s) * np.floor(h / s)  # lower bound on points inside the box
    rows.append({"stride": s, "median points": np.median(pts), "share with 0 points": (pts == 0).mean(),
                 "share with <4 points": (pts < 4).mean()})
display(pd.DataFrame(rows).style.format("{:.3f}", subset=["share with 0 points", "share with <4 points"]))

If many boxes have zero or one point even on stride 8, the assigner cannot pick its top-k candidates for them, and in dense scenes neighbouring boxes compete for the same points. A stride-4 P2 head (Y1) quadruples the number of candidate points per box; higher resolution (Y2) has a similar effect on every level, at a quadratic compute cost.

## 3. Experiments

In [ ]:
BASE = dict(model="yolov8s.pt", imgsz=P["imgsz"], epochs=P["epochs"], batch=8, seed=P["seed"])
EXPERIMENTS = {
    "yolo_y0_base": BASE,
    "yolo_y1_p2": {**BASE, "model": "yolov8s-p2.yaml", "weights": "yolov8s.pt"},
}
ARCH = "yolo_y1_p2"    # set to the better of y0/y1 before running Y2/Y3
for s in [1024, 1536]:
    EXPERIMENTS[f"yolo_y2_imgsz{s}"] = {**EXPERIMENTS[ARCH], "imgsz": s, "batch": 8 if s <= 1280 else 4}
EXPERIMENTS["yolo_y3_tal13"] = {**EXPERIMENTS[ARCH], "tal_topk": 13}
pd.DataFrame(EXPERIMENTS).T

In [ ]:
# Accelerator "GPU T4 x2": two runs train at the same time, one per GPU (quota is per session hour).
# Queue as many runs as fit into the 12 h session; finished runs are skipped. Logs: runs/<name>.log
RUN = ["yolo_y0_base", "yolo_y1_p2"]
codes = parallel.run_parallel([(n, EXPERIMENTS[n]) for n in RUN], RUNS, folds_csv=OUT / "folds.csv", data_yaml=data_yaml)

**Search range and selection.** Architecture: {P3–P5, P2–P5}. Resolution: {1024, 1280, 1536} (above 1536 the batch no longer fits a T4 at a useful size). Label assignment: TAL top-k ∈ {10, 13}. Each choice is made on the fold-0 holdout before moving to the next.

## 4. Results by object size, against the tuned anchor-based detectors

In [ ]:
COLS = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_max", "ms_per_img", "epochs"]
def show(res, cols=COLS):
    cols = [c for c in cols if c in res]
    num = [c for c in cols if c not in ("ms_per_img", "epochs")]
    return res[cols].style.format("{:.4f}", subset=num).format("{:.1f}", subset=[c for c in ["ms_per_img"] if c in cols])

res = vdet.collect_results(RESULT_DIRS, "yolo")
others = [vdet.collect_results(RESULT_DIRS, p) for p in ("frcnn", "retina")]
for o in others:
    if len(o): res = pd.concat([res, o.loc[[o.AP.idxmax()]]])
if len(res):
    display(show(res))
    res[["AP_small", "AP_medium", "AP_large"]].plot.bar(figsize=(12, 4), rot=30, title="AP by object size")
    plt.tight_layout(); plt.savefig(OUT / "ap_by_size_anchor_free.png", dpi=120); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for run in [r for r in res.index if r.startswith("yolo")] if len(res) else []:
    h = vdet.load_history(RESULT_DIRS, run)
    if h is not None: ax.plot(h.epoch, h.AP_ultra, "o-", label=run)
ax.set_title("Ultralytics val mAP50-95 per epoch (masked val images)"); ax.set_xlabel("epoch"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

Note: per-epoch curves come from the Ultralytics validator on the exported (masked) val images; the tables use our evaluator on the original frames, as for every other model.

## 5. Export and test submission

In [ ]:
BEST = res.drop(index=[i for i in res.index if not i.startswith("yolo")]).AP.idxmax() if len(res) else "yolo_y0_base"
model = ultra.load(RUNS / BEST); imgsz = json.loads((RUNS / BEST / "config.json").read_text())["imgsz"]
test_preds = ultra.predict_test(model, ROOT, imgsz)
test_preds.to_csv(OUT / f"test_preds_{BEST}.csv.gz", index=False)
vdet.write_submission(test_preds, ROOT / "test_images.csv", OUT / f"submission_{BEST}.csv")

In [ ]:
# Results for the repo: unzip results.zip from the notebook Output into <repo>/results/ and commit.
vdet.export_results(RUNS, OUT / "results", preds_for=(BEST,))